In [ ]:
"""
Risk-aware ore-waste classification for tailings re-mining using drift learning and conditional geostatistical simulation.

GCNN-RNN-SGSIM workflow for risk-aware ore-waste classification at block support.

This script implements a hybrid modelling workflow in which systematic spatial structure is
learned with a GCNN-RNN drift model and the remaining uncertainty is simulated with SGSIM.
The workflow is designed for block-support grade modelling, uncertainty quantification, and
decision-oriented outputs derived from conditional realizations.

Author:      Keyumars Anvari
Supervisor:  Professor Jörg Benndorf
Affiliation: Institute of Mine Surveying and Geodesy,
             TU Bergakademie Freiberg, Freiberg, Germany

Main steps
----------
1. Read sample, block, and optional zone inputs.
2. Prepare coordinates, grades, and zone labels.
3. Transform grades to log and normal-score space.
4. Train a leakage-safe drift model using borehole-wise grouped splits.
5. Model residual spatial continuity with variogram-based simulation.
6. Generate conditional realizations on the block grid.
7. Back-transform realizations and compute block-wise summaries.
8. Export reproducible outputs for further analysis.

Expected sample columns
-----------------------
x, y, z, grade, borehole
Optional: zone

Expected block columns
----------------------
x, y, z
Optional: zone

Expected zone columns
---------------------
x, y, z, zone

Outputs
-------
summary.csv
block_estimates.csv
realizations.npy
config.json

Notes
-----
The script can run either from user-supplied input files or from an internal fallback dataset
when file paths are not provided. The fallback path is included only to keep the workflow
directly runnable.
"""

from __future__ import annotations

import argparse
import json
import math
import os
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, Optional, Tuple

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, log_loss, mean_absolute_error, r2_score, roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")

if not hasattr(math, "erfinv"):
    def _erfinv_approx(x: float) -> float:
        a = 0.147
        x = min(max(x, -0.999999), 0.999999)
        ln = math.log(1.0 - x * x)
        t = 2.0 / (math.pi * a) + ln / 2.0
        inner = max(t * t - ln / a, 0.0)
        return math.copysign(math.sqrt(max(math.sqrt(inner) - t, 0.0)), x)

    math.erfinv = _erfinv_approx  # type: ignore[attr-defined]

try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
except Exception as exc:
    raise RuntimeError(
        "TensorFlow is required to run this script. Install tensorflow before execution."
    ) from exc


@dataclass
class Config:
    sample_path: str = ""
    block_path: str = ""
    zone_path: str = ""
    output_dir: str = "results"
    seed: int = 7

    n_realizations: int = 50
    knn_k: int = 12
    sim_k: int = 12
    sim_candidates: int = 40
    anisotropy_z: float = 3.0

    learning_rate: float = 8e-4
    epochs: int = 90
    batch_size: int = 64
    patience: int = 12
    drift_ensemble: int = 4
    n_facies: int = 4

    cutoff_grade: Optional[float] = None
    price_per_kg: float = 9.0
    recovery: float = 0.85
    mining_cost: float = 12.0
    processing_cost: float = 18.0

    fallback_samples: int = 420
    fallback_grid_x: int = 18
    fallback_grid_y: int = 14
    fallback_grid_z: int = 6


def ensure_dir(path: str) -> None:
    Path(path).mkdir(parents=True, exist_ok=True)


def set_seed(seed: int) -> None:
    np.random.seed(seed)
    tf.random.set_seed(seed)


def scaled_xyz(xyz: np.ndarray, anisotropy_z: float) -> np.ndarray:
    out = np.asarray(xyz, dtype=float).copy()
    out[:, 2] *= float(anisotropy_z)
    return out


def economic_cutoff(cfg: Config) -> float:
    if cfg.cutoff_grade is not None:
        return float(cfg.cutoff_grade)
    total_cost = cfg.mining_cost + cfg.processing_cost
    return float(total_cost / (cfg.price_per_kg * cfg.recovery + 1e-12) * 1e3)


def profit_per_tonne(grade_ppm: np.ndarray, cfg: Config) -> np.ndarray:
    grade_ppm = np.asarray(grade_ppm, dtype=float)
    revenue = (grade_ppm * 1e-3) * cfg.price_per_kg * cfg.recovery
    cost = cfg.mining_cost + cfg.processing_cost
    return revenue - cost


def normal_score_fit(values: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    values = np.asarray(values, dtype=float).ravel()
    xs = np.sort(values)
    p = (np.arange(1, len(xs) + 1) - 0.5) / max(len(xs), 1)
    zs = np.sqrt(2.0) * np.vectorize(math.erfinv)(2.0 * p - 1.0)
    return xs, zs


def normal_score_transform(values: np.ndarray, xs: np.ndarray, zs: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=float).ravel()
    return np.interp(values, xs, zs)


def inverse_normal_score(values: np.ndarray, xs: np.ndarray, zs: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=float).ravel()
    return np.interp(values, zs, xs)


def _build_default_inputs(cfg: Config) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    rng = np.random.RandomState(cfg.seed)

    gx = np.arange(cfg.fallback_grid_x, dtype=float) * 10.0
    gy = np.arange(cfg.fallback_grid_y, dtype=float) * 10.0
    gz = np.arange(cfg.fallback_grid_z, dtype=float) * 3.0
    xx, yy, zz = np.meshgrid(gx, gy, gz, indexing="ij")
    xyz = np.column_stack([xx.ravel(), yy.ravel(), zz.ravel()])

    base_signal = (
        1.4 * np.sin(xyz[:, 0] / 28.0)
        + 1.1 * np.cos(xyz[:, 1] / 20.0)
        - 0.8 * xyz[:, 2] / (max(gz.max(), 1.0) + 1.0)
    )
    zone_signal = np.sin(xyz[:, 0] / 25.0) + np.cos(xyz[:, 1] / 18.0) - 0.5 * xyz[:, 2] / max(gz.max(), 1.0)
    zone = np.digitize(zone_signal, bins=np.quantile(zone_signal, [0.33, 0.66])).astype(int)

    latent_grade = 900.0 + 220.0 * base_signal + 70.0 * zone + rng.normal(0.0, 80.0, size=len(xyz))
    latent_grade = np.clip(latent_grade, 60.0, None)

    blocks = pd.DataFrame(xyz, columns=["x", "y", "z"])
    blocks["zone"] = zone
    blocks["latent_truth"] = latent_grade

    picked = rng.choice(len(blocks), size=min(cfg.fallback_samples, len(blocks)), replace=False)
    samples = blocks.loc[picked, ["x", "y", "z", "zone"]].copy().reset_index(drop=True)
    samples["grade"] = np.clip(
        blocks.loc[picked, "latent_truth"].to_numpy() + rng.normal(0.0, 45.0, size=len(picked)),
        60.0,
        None,
    )
    n_boreholes = max(12, len(samples) // 22)
    samples["borehole"] = [f"BH_{idx:03d}" for idx in rng.randint(0, n_boreholes, size=len(samples))]

    zones = blocks[["x", "y", "z", "zone"]].copy()
    return samples, blocks[["x", "y", "z", "zone", "latent_truth"]].copy(), zones


def load_inputs(cfg: Config) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    if cfg.sample_path and cfg.block_path:
        samples = pd.read_csv(cfg.sample_path)
        blocks = pd.read_csv(cfg.block_path)

        if cfg.zone_path:
            zones = pd.read_csv(cfg.zone_path)
        elif "zone" in blocks.columns:
            zones = blocks[["x", "y", "z", "zone"]].copy()
        else:
            zones = blocks[["x", "y", "z"]].copy()
            zones["zone"] = 0

        return samples, blocks, zones

    return _build_default_inputs(cfg)


def attach_missing_zones(
    samples: pd.DataFrame,
    blocks: pd.DataFrame,
    zones: pd.DataFrame,
    cfg: Config,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    zone_xyz = zones[["x", "y", "z"]].to_numpy(float)
    zone_labels = zones["zone"].to_numpy()

    nn = NearestNeighbors(n_neighbors=1)
    nn.fit(scaled_xyz(zone_xyz, cfg.anisotropy_z))

    if "zone" not in samples.columns:
        _, idx = nn.kneighbors(scaled_xyz(samples[["x", "y", "z"]].to_numpy(float), cfg.anisotropy_z))
        samples = samples.copy()
        samples["zone"] = zone_labels[idx[:, 0]]

    if "zone" not in blocks.columns:
        _, idx = nn.kneighbors(scaled_xyz(blocks[["x", "y", "z"]].to_numpy(float), cfg.anisotropy_z))
        blocks = blocks.copy()
        blocks["zone"] = zone_labels[idx[:, 0]]

    return samples, blocks


def recode_zones(
    samples: pd.DataFrame,
    blocks: pd.DataFrame,
    zones: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    values = pd.concat(
        [samples["zone"], blocks["zone"], zones["zone"]],
        ignore_index=True,
    ).astype(str)
    mapping = {value: idx for idx, value in enumerate(sorted(values.unique()))}

    samples = samples.copy()
    blocks = blocks.copy()
    zones = zones.copy()

    samples["zone"] = samples["zone"].astype(str).map(mapping).astype(int)
    blocks["zone"] = blocks["zone"].astype(str).map(mapping).astype(int)
    zones["zone"] = zones["zone"].astype(str).map(mapping).astype(int)

    return samples, blocks, zones


def knn_idw_predict(
    xyz_ref: np.ndarray,
    y_ref: np.ndarray,
    xyz_target: np.ndarray,
    cfg: Config,
    k: int,
    exclude_self: bool = False,
) -> np.ndarray:
    x_ref = scaled_xyz(np.asarray(xyz_ref, dtype=float), cfg.anisotropy_z)
    x_target = scaled_xyz(np.asarray(xyz_target, dtype=float), cfg.anisotropy_z)
    y_ref = np.asarray(y_ref, dtype=float).ravel()

    n_neighbors = min(len(x_ref), int(k) + (1 if exclude_self else 0))
    nn = NearestNeighbors(n_neighbors=n_neighbors)
    nn.fit(x_ref)
    distances, indices = nn.kneighbors(x_target, return_distance=True)

    if exclude_self and distances.shape[1] > 1:
        distances = distances[:, 1:]
        indices = indices[:, 1:]

    weights = 1.0 / np.maximum(distances, 1e-12)
    return np.sum(weights * y_ref[indices], axis=1) / (np.sum(weights, axis=1) + 1e-12)


def fit_facies(df: pd.DataFrame, cfg: Config) -> Tuple[np.ndarray, KMeans, StandardScaler]:
    features = np.column_stack([
        np.log1p(df["x"].to_numpy(float) - df["x"].min() + 1.0),
        np.log1p(df["y"].to_numpy(float) - df["y"].min() + 1.0),
        np.log1p(df["z"].to_numpy(float) - df["z"].min() + 1.0),
        df["zone"].to_numpy(float),
    ])
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)
    model = KMeans(n_clusters=cfg.n_facies, n_init=20, random_state=cfg.seed)
    labels = model.fit_predict(features_scaled)
    return labels, model, scaler


def predict_facies(df: pd.DataFrame, model: KMeans, scaler: StandardScaler) -> np.ndarray:
    features = np.column_stack([
        np.log1p(df["x"].to_numpy(float) - df["x"].min() + 1.0),
        np.log1p(df["y"].to_numpy(float) - df["y"].min() + 1.0),
        np.log1p(df["z"].to_numpy(float) - df["z"].min() + 1.0),
        df["zone"].to_numpy(float),
    ])
    return model.predict(scaler.transform(features))


def one_hot(labels: np.ndarray, n_classes: int) -> np.ndarray:
    labels = np.asarray(labels, dtype=int).ravel()
    out = np.zeros((len(labels), n_classes), dtype=float)
    out[np.arange(len(labels)), labels] = 1.0
    return out


def zone_soft_weights(target_xyz: np.ndarray, zone_cloud: pd.DataFrame, n_zones: int, cfg: Config) -> np.ndarray:
    zone_xyz = zone_cloud[["x", "y", "z"]].to_numpy(float)
    zone_labels = zone_cloud["zone"].to_numpy(int)

    n_neighbors = min(len(zone_cloud), max(8, cfg.knn_k * 2))
    nn = NearestNeighbors(n_neighbors=n_neighbors)
    nn.fit(scaled_xyz(zone_xyz, cfg.anisotropy_z))
    distances, indices = nn.kneighbors(scaled_xyz(target_xyz, cfg.anisotropy_z), return_distance=True)

    positive = distances[distances > 0]
    scale = max(float(np.median(positive)) if positive.size else 1.0, 1e-6)
    weights = np.exp(-distances / scale)

    out = np.zeros((len(target_xyz), n_zones), dtype=float)
    nearby = zone_labels[indices]
    for zone_id in range(n_zones):
        out[:, zone_id] = np.sum(weights * (nearby == zone_id), axis=1)

    out /= np.sum(out, axis=1, keepdims=True) + 1e-12
    return out


def build_sequence_features(
    target_xyz: np.ndarray,
    ref_xyz: np.ndarray,
    ref_grade_log: np.ndarray,
    base_prediction: np.ndarray,
    facies_one_hot: np.ndarray,
    zone_weights: np.ndarray,
    cfg: Config,
    exclude_self: bool,
) -> Tuple[np.ndarray, np.ndarray]:
    x_ref = scaled_xyz(ref_xyz, cfg.anisotropy_z)
    x_target = scaled_xyz(target_xyz, cfg.anisotropy_z)
    n_neighbors = min(len(ref_xyz), cfg.knn_k + (1 if exclude_self else 0))

    nn = NearestNeighbors(n_neighbors=n_neighbors)
    nn.fit(x_ref)
    distances, indices = nn.kneighbors(x_target, return_distance=True)

    if exclude_self and distances.shape[1] > 1:
        distances = distances[:, 1:]
        indices = indices[:, 1:]

    neigh_xyz = ref_xyz[indices]
    dx = neigh_xyz[:, :, 0] - target_xyz[:, None, 0]
    dy = neigh_xyz[:, :, 1] - target_xyz[:, None, 1]
    dz = neigh_xyz[:, :, 2] - target_xyz[:, None, 2]
    neigh_grade = ref_grade_log[indices]

    sequence = np.stack([neigh_grade, dx, dy, dz, distances], axis=-1)
    static = np.column_stack([
        base_prediction,
        target_xyz[:, 0],
        target_xyz[:, 1],
        target_xyz[:, 2],
        facies_one_hot,
        zone_weights,
        np.mean(neigh_grade, axis=1),
        np.std(neigh_grade, axis=1),
    ])
    return sequence, static


class AttentionPool(layers.Layer):
    def __init__(self) -> None:
        super().__init__()
        self.score = layers.Dense(1)

    def call(self, inputs, training=None):
        weights = tf.nn.softmax(self.score(inputs), axis=1)
        return tf.reduce_sum(inputs * weights, axis=1)


def build_drift_model(n_seq_features: int, n_static_features: int, cfg: Config) -> keras.Model:
    seq_input = keras.Input(shape=(cfg.knn_k, n_seq_features), name="sequence")
    static_input = keras.Input(shape=(n_static_features,), name="static")

    seq_branch = layers.Dense(96, activation="relu")(seq_input)
    seq_branch = layers.Dense(64, activation="relu")(seq_branch)
    seq_branch = layers.Bidirectional(layers.GRU(32, return_sequences=True))(seq_branch)
    seq_branch = AttentionPool()(seq_branch)

    static_branch = layers.Dense(64, activation="relu")(static_input)
    static_branch = layers.Dense(32, activation="relu")(static_branch)

    merged = layers.Concatenate()([seq_branch, static_branch])
    merged = layers.Dense(96, activation="relu")(merged)
    merged = layers.Dropout(0.15)(merged)
    merged = layers.Dense(48, activation="relu")(merged)
    output = layers.Dense(1, name="delta")(merged)

    model = keras.Model(inputs=[seq_input, static_input], outputs=output)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=cfg.learning_rate),
        loss=keras.losses.Huber(delta=0.4),
    )
    return model


def fit_global_variogram(xyz: np.ndarray, values: np.ndarray, cfg: Config) -> Tuple[float, float, float]:
    rng = np.random.RandomState(cfg.seed)
    xyz_scaled = scaled_xyz(np.asarray(xyz, dtype=float), cfg.anisotropy_z)
    values = np.asarray(values, dtype=float).ravel()

    n = len(values)
    n_pairs = min(25000, max(n * 10, 1000))
    i = rng.randint(0, n, size=n_pairs)
    j = rng.randint(0, n, size=n_pairs)
    mask = i != j
    i, j = i[mask], j[mask]

    distances = np.linalg.norm(xyz_scaled[i] - xyz_scaled[j], axis=1)
    sill = float(np.var(values))
    nugget = max(0.02 * sill, 1e-6)
    range_value = float(np.quantile(distances[distances > 0], 0.55)) if np.any(distances > 0) else 1.0
    return nugget, max(sill, nugget + 1e-6), max(range_value, 1e-6)


def covariance_exponential(distance: np.ndarray, nugget: float, sill: float, range_value: float) -> np.ndarray:
    partial_sill = max(sill - nugget, 1e-12)
    return partial_sill * np.exp(-np.asarray(distance, dtype=float) / max(range_value, 1e-12))


def ordinary_kriging_mean_var(
    target_xyz: np.ndarray,
    neighbor_xyz: np.ndarray,
    neighbor_values: np.ndarray,
    variogram: Tuple[float, float, float],
) -> Tuple[float, float]:
    nugget, sill, range_value = variogram
    n_neighbors = len(neighbor_values)

    if n_neighbors == 0:
        return 0.0, sill

    if n_neighbors == 1:
        d0 = np.linalg.norm(target_xyz - neighbor_xyz[0])
        c0 = covariance_exponential(np.array([d0]), nugget, sill, range_value)[0]
        weight = c0 / max(sill, 1e-12)
        mean = weight * float(neighbor_values[0])
        var = max(sill - weight * c0, 1e-12)
        return float(mean), float(var)

    d_nn = np.linalg.norm(neighbor_xyz[:, None, :] - neighbor_xyz[None, :, :], axis=2)
    d_target = np.linalg.norm(neighbor_xyz - target_xyz[None, :], axis=1)

    cov_nn = covariance_exponential(d_nn, nugget, sill, range_value)
    np.fill_diagonal(cov_nn, sill)
    cov_target = covariance_exponential(d_target, nugget, sill, range_value)

    system = np.zeros((n_neighbors + 1, n_neighbors + 1), dtype=float)
    system[:n_neighbors, :n_neighbors] = cov_nn
    system[:n_neighbors, -1] = 1.0
    system[-1, :n_neighbors] = 1.0

    rhs = np.zeros(n_neighbors + 1, dtype=float)
    rhs[:n_neighbors] = cov_target
    rhs[-1] = 1.0

    try:
        solution = np.linalg.solve(system, rhs)
    except np.linalg.LinAlgError:
        solution, *_ = np.linalg.lstsq(system, rhs, rcond=None)

    weights = solution[:n_neighbors]
    lagrange = solution[-1]
    mean = float(np.dot(weights, neighbor_values))
    var = float(max(sill - np.dot(weights, cov_target) - lagrange, 1e-12))
    return mean, var


def precompute_candidates(sample_xyz: np.ndarray, block_xyz: np.ndarray, cfg: Config) -> np.ndarray:
    combined = np.vstack([sample_xyz, block_xyz])
    combined_scaled = scaled_xyz(combined, cfg.anisotropy_z)
    nn = NearestNeighbors(n_neighbors=min(len(combined), cfg.sim_candidates + 1))
    nn.fit(combined_scaled)
    return nn.kneighbors(combined_scaled[len(sample_xyz):], return_distance=False)[:, 1:]


def sgsim(
    conditioning_ns: np.ndarray,
    sample_xyz: np.ndarray,
    block_xyz: np.ndarray,
    variogram: Tuple[float, float, float],
    candidates: np.ndarray,
    cfg: Config,
    seed: int,
) -> np.ndarray:
    rng = np.random.RandomState(seed)
    combined_xyz = scaled_xyz(np.vstack([sample_xyz, block_xyz]), cfg.anisotropy_z)
    values = np.zeros(len(combined_xyz), dtype=float)
    known = np.zeros(len(combined_xyz), dtype=bool)

    n_samples = len(sample_xyz)
    values[:n_samples] = conditioning_ns
    known[:n_samples] = True

    order = np.arange(len(block_xyz))
    rng.shuffle(order)

    for block_id in order:
        candidate_ids = candidates[block_id]
        candidate_ids = candidate_ids[known[candidate_ids]]
        candidate_ids = candidate_ids[: cfg.sim_k]

        if len(candidate_ids) > 0:
            mean, var = ordinary_kriging_mean_var(
                combined_xyz[n_samples + block_id],
                combined_xyz[candidate_ids],
                values[candidate_ids],
                variogram,
            )
        else:
            mean, var = 0.0, variogram[1]

        values[n_samples + block_id] = mean + math.sqrt(max(var, 1e-12)) * rng.randn()
        known[n_samples + block_id] = True

    return values[n_samples:]


def crps_ensemble(draws: np.ndarray, truth: np.ndarray) -> np.ndarray:
    draws = np.asarray(draws, dtype=float)
    truth = np.asarray(truth, dtype=float).ravel()
    n_realizations = draws.shape[0]

    term1 = np.mean(np.abs(draws - truth[None, :]), axis=0)
    draws_sorted = np.sort(draws, axis=0)
    coeff = (2 * np.arange(1, n_realizations + 1) - n_realizations - 1).reshape(-1, 1)
    term2 = np.sum(coeff * draws_sorted, axis=0) / float(n_realizations ** 2)
    return term1 - term2


def summarize_probabilistic_skill(draws_ppm: np.ndarray, truth_ppm: np.ndarray, cutoff: float) -> Dict[str, float]:
    draws_log = np.log10(np.maximum(draws_ppm, 1e-6))
    truth_log = np.log10(np.maximum(truth_ppm, 1e-6))
    crps = crps_ensemble(draws_log, truth_log)

    p10 = np.quantile(draws_log, 0.10, axis=0)
    p90 = np.quantile(draws_log, 0.90, axis=0)
    coverage = float(np.mean((truth_log >= p10) & (truth_log <= p90)))
    width = float(np.mean(p90 - p10))

    y = (truth_ppm >= cutoff).astype(int)
    p = np.clip(np.mean(draws_ppm >= cutoff, axis=0), 1e-8, 1.0 - 1e-8)

    out: Dict[str, float] = {
        "crps_mean": float(np.mean(crps)),
        "coverage_p10_p90": coverage,
        "width_p10_p90": width,
        "brier": float(np.mean((p - y) ** 2)),
        "logloss": float(log_loss(y, p, labels=[0, 1])),
        "roc_auc": float("nan"),
        "pr_auc": float("nan"),
    }

    if len(np.unique(y)) == 2:
        out["roc_auc"] = float(roc_auc_score(y, p))
        out["pr_auc"] = float(average_precision_score(y, p))

    return out


def probability_of_ore(draws_ppm: np.ndarray, cutoff: float) -> np.ndarray:
    return np.mean(np.asarray(draws_ppm, dtype=float) >= float(cutoff), axis=0)


def select_threshold_from_training(probabilities: np.ndarray, truth_labels: np.ndarray) -> float:
    best_threshold = 0.5
    best_score = None

    for threshold in np.linspace(0.05, 0.95, 181):
        predicted = (probabilities >= threshold).astype(int)
        fp = np.sum((truth_labels == 0) & (predicted == 1))
        fn = np.sum((truth_labels == 1) & (predicted == 0))
        score = (
            2.0 * fn + 1.0 * fp,
            float(np.mean(predicted != truth_labels)),
            int(abs(np.sum(predicted) - np.sum(truth_labels))),
        )
        if best_score is None or score < best_score:
            best_score = score
            best_threshold = float(threshold)

    return best_threshold


def zone_variograms(samples: pd.DataFrame, residual_ns: np.ndarray, cfg: Config) -> Dict[int, Tuple[float, float, float]]:
    xyz = samples[["x", "y", "z"]].to_numpy(float)
    global_model = fit_global_variogram(xyz, residual_ns, cfg)
    models: Dict[int, Tuple[float, float, float]] = {}

    for zone_id in sorted(samples["zone"].unique()):
        mask = samples["zone"].to_numpy(int) == int(zone_id)
        if mask.sum() < max(20, cfg.knn_k * 2):
            models[int(zone_id)] = global_model
        else:
            models[int(zone_id)] = fit_global_variogram(xyz[mask], residual_ns[mask], cfg)

    return models


def combine_zone_variograms(zone_weights: np.ndarray, models: Dict[int, Tuple[float, float, float]]) -> np.ndarray:
    ordered = np.array([models[z] for z in sorted(models)], dtype=float)
    return zone_weights @ ordered


def grouped_drift_oof(samples: pd.DataFrame, zone_cloud: pd.DataFrame, cfg: Config) -> np.ndarray:
    xyz = samples[["x", "y", "z"]].to_numpy(float)
    y = samples["grade_log"].to_numpy(float)
    groups = samples["borehole"].astype(str).to_numpy()

    facies_labels, _, _ = fit_facies(samples, cfg)
    facies_oh = one_hot(facies_labels, cfg.n_facies)
    zone_weights = zone_soft_weights(xyz, zone_cloud, int(zone_cloud["zone"].nunique()), cfg)

    if len(np.unique(groups)) < 2:
        groups = np.array([f"G_{i:03d}" for i in range(len(groups))])

    splitter = GroupKFold(n_splits=min(5, len(np.unique(groups))))
    oof = np.zeros(len(samples), dtype=float)

    for fold, (train_idx, test_idx) in enumerate(splitter.split(xyz, y, groups)):
        x_train = xyz[train_idx]
        x_test = xyz[test_idx]
        y_train = y[train_idx]

        base_train = knn_idw_predict(x_train, y_train, x_train, cfg, cfg.knn_k, exclude_self=True)
        base_test = knn_idw_predict(x_train, y_train, x_test, cfg, cfg.knn_k, exclude_self=False)

        seq_train, stat_train = build_sequence_features(
            x_train,
            x_train,
            y_train,
            base_train,
            facies_oh[train_idx],
            zone_weights[train_idx],
            cfg,
            exclude_self=True,
        )
        seq_test, stat_test = build_sequence_features(
            x_test,
            x_train,
            y_train,
            base_test,
            facies_oh[test_idx],
            zone_weights[test_idx],
            cfg,
            exclude_self=False,
        )

        scaler = StandardScaler()
        stat_train = scaler.fit_transform(stat_train)
        stat_test = scaler.transform(stat_test)
        target = (y_train - base_train).reshape(-1, 1)

        train_groups = groups[train_idx]
        unique_groups = np.unique(train_groups)
        rng = np.random.RandomState(cfg.seed + fold)
        rng.shuffle(unique_groups)
        n_valid = max(1, int(0.2 * len(unique_groups)))
        valid_groups = set(unique_groups[:n_valid])

        fold_train = np.array([g not in valid_groups for g in train_groups])
        fold_valid = ~fold_train

        predictions = []
        for member in range(cfg.drift_ensemble):
            set_seed(cfg.seed + 1000 * fold + 29 * member)
            model = build_drift_model(seq_train.shape[-1], stat_train.shape[-1], cfg)
            stopper = keras.callbacks.EarlyStopping(
                monitor="val_loss",
                patience=cfg.patience,
                restore_best_weights=True,
                verbose=0,
            )
            model.fit(
                [seq_train[fold_train], stat_train[fold_train]],
                target[fold_train],
                validation_data=([seq_train[fold_valid], stat_train[fold_valid]], target[fold_valid]),
                epochs=cfg.epochs,
                batch_size=cfg.batch_size,
                verbose=0,
                callbacks=[stopper],
            )
            delta = model.predict([seq_test, stat_test], verbose=0).ravel()
            predictions.append(base_test + delta)
            keras.backend.clear_session()

        oof[test_idx] = np.mean(np.vstack(predictions), axis=0)

    return oof


def fit_final_drift(
    samples: pd.DataFrame,
    blocks: pd.DataFrame,
    zone_cloud: pd.DataFrame,
    cfg: Config,
) -> Tuple[np.ndarray, np.ndarray]:
    sample_xyz = samples[["x", "y", "z"]].to_numpy(float)
    block_xyz = blocks[["x", "y", "z"]].to_numpy(float)
    y = samples["grade_log"].to_numpy(float)

    facies_labels, facies_model, facies_scaler = fit_facies(samples, cfg)
    facies_train = one_hot(facies_labels, cfg.n_facies)
    facies_block = one_hot(predict_facies(blocks, facies_model, facies_scaler), cfg.n_facies)

    n_zones = int(zone_cloud["zone"].nunique())
    zone_train = zone_soft_weights(sample_xyz, zone_cloud, n_zones, cfg)
    zone_block = zone_soft_weights(block_xyz, zone_cloud, n_zones, cfg)

    base_train = knn_idw_predict(sample_xyz, y, sample_xyz, cfg, cfg.knn_k, exclude_self=True)
    base_block = knn_idw_predict(sample_xyz, y, block_xyz, cfg, cfg.knn_k, exclude_self=False)

    seq_train, stat_train = build_sequence_features(
        sample_xyz,
        sample_xyz,
        y,
        base_train,
        facies_train,
        zone_train,
        cfg,
        exclude_self=True,
    )
    seq_block, stat_block = build_sequence_features(
        block_xyz,
        sample_xyz,
        y,
        base_block,
        facies_block,
        zone_block,
        cfg,
        exclude_self=False,
    )

    scaler = StandardScaler()
    stat_train = scaler.fit_transform(stat_train)
    stat_block = scaler.transform(stat_block)
    target = (y - base_train).reshape(-1, 1)

    sample_members = []
    block_members = []

    for member in range(cfg.drift_ensemble):
        set_seed(cfg.seed + 5000 + 37 * member)
        model = build_drift_model(seq_train.shape[-1], stat_train.shape[-1], cfg)
        stopper = keras.callbacks.EarlyStopping(
            monitor="loss",
            patience=cfg.patience,
            restore_best_weights=True,
            verbose=0,
        )
        model.fit(
            [seq_train, stat_train],
            target,
            epochs=cfg.epochs,
            batch_size=cfg.batch_size,
            verbose=0,
            callbacks=[stopper],
        )
        sample_delta = model.predict([seq_train, stat_train], verbose=0).ravel()
        block_delta = model.predict([seq_block, stat_block], verbose=0).ravel()

        sample_members.append(base_train + sample_delta)
        block_members.append(base_block + block_delta)
        keras.backend.clear_session()

    return np.mean(np.vstack(sample_members), axis=0), np.mean(np.vstack(block_members), axis=0)


def simulate_hybrid(
    samples: pd.DataFrame,
    blocks: pd.DataFrame,
    drift_sample_log: np.ndarray,
    drift_block_log: np.ndarray,
    zone_weights_block: np.ndarray,
    cfg: Config,
) -> np.ndarray:
    residual = samples["grade_log"].to_numpy(float) - drift_sample_log
    xs, zs = normal_score_fit(residual)
    residual_ns = normal_score_transform(residual, xs, zs)

    sample_xyz = samples[["x", "y", "z"]].to_numpy(float)
    block_xyz = blocks[["x", "y", "z"]].to_numpy(float)
    candidates = precompute_candidates(sample_xyz, block_xyz, cfg)

    models_by_zone = zone_variograms(samples, residual_ns, cfg)
    mixed_models = combine_zone_variograms(zone_weights_block, models_by_zone)

    combined_xyz = scaled_xyz(np.vstack([sample_xyz, block_xyz]), cfg.anisotropy_z)
    n_samples = len(sample_xyz)
    realizations = []

    for r in range(cfg.n_realizations):
        rng = np.random.RandomState(cfg.seed + 20_000 + r)
        values = np.zeros(len(combined_xyz), dtype=float)
        known = np.zeros(len(combined_xyz), dtype=bool)

        values[:n_samples] = residual_ns
        known[:n_samples] = True

        order = np.arange(len(block_xyz))
        rng.shuffle(order)

        for block_id in order:
            candidate_ids = candidates[block_id]
            candidate_ids = candidate_ids[known[candidate_ids]]
            candidate_ids = candidate_ids[: cfg.sim_k]

            if len(candidate_ids) > 0:
                mean, var = ordinary_kriging_mean_var(
                    combined_xyz[n_samples + block_id],
                    combined_xyz[candidate_ids],
                    values[candidate_ids],
                    tuple(mixed_models[block_id]),
                )
            else:
                mean, var = 0.0, float(mixed_models[block_id, 1])

            values[n_samples + block_id] = mean + math.sqrt(max(var, 1e-12)) * rng.randn()
            known[n_samples + block_id] = True

        residual_sim = inverse_normal_score(values[n_samples:], xs, zs)
        grade_log = drift_block_log + residual_sim
        realizations.append(np.clip(10.0 ** grade_log, 1e-6, None))

    return np.asarray(realizations, dtype=np.float32)


def run(cfg: Config) -> None:
    ensure_dir(cfg.output_dir)
    set_seed(cfg.seed)

    samples, blocks, zone_cloud = load_inputs(cfg)
    samples, blocks = attach_missing_zones(samples, blocks, zone_cloud, cfg)
    samples, blocks, zone_cloud = recode_zones(samples, blocks, zone_cloud)

    samples = samples.copy()
    blocks = blocks.copy()
    samples["grade_log"] = np.log10(np.maximum(samples["grade"].to_numpy(float), 1e-6))

    cutoff = economic_cutoff(cfg)
    sample_xyz = samples[["x", "y", "z"]].to_numpy(float)
    block_xyz = blocks[["x", "y", "z"]].to_numpy(float)

    grade_xs, grade_zs = normal_score_fit(samples["grade_log"].to_numpy(float))
    grade_ns = normal_score_transform(samples["grade_log"].to_numpy(float), grade_xs, grade_zs)

    baseline_variogram = fit_global_variogram(sample_xyz, grade_ns, cfg)
    baseline_candidates = precompute_candidates(sample_xyz, block_xyz, cfg)

    sgsim_realizations = []
    for r in range(cfg.n_realizations):
        sim_ns = sgsim(
            grade_ns,
            sample_xyz,
            block_xyz,
            baseline_variogram,
            baseline_candidates,
            cfg,
            seed=cfg.seed + 10_000 + r,
        )
        sim_log = inverse_normal_score(sim_ns, grade_xs, grade_zs)
        sgsim_realizations.append(np.clip(10.0 ** sim_log, 1e-6, None))

    sgsim_realizations = np.asarray(sgsim_realizations, dtype=np.float32)

    drift_oof = grouped_drift_oof(samples, zone_cloud, cfg)
    drift_sample, drift_block = fit_final_drift(samples, blocks, zone_cloud, cfg)

    n_zones = int(zone_cloud["zone"].nunique())
    zone_weights_block = zone_soft_weights(block_xyz, zone_cloud, n_zones, cfg)
    hybrid_realizations = simulate_hybrid(samples, blocks, drift_sample, drift_block, zone_weights_block, cfg)

    block_estimates = blocks[["x", "y", "z", "zone"]].copy()
    block_estimates["sgsim_mean"] = np.mean(sgsim_realizations, axis=0)
    block_estimates["gcnn_rnn_sgsim_mean"] = np.mean(hybrid_realizations, axis=0)
    block_estimates["p_ore_sgsim"] = probability_of_ore(sgsim_realizations, cutoff)
    block_estimates["p_ore_gcnn_rnn_sgsim"] = probability_of_ore(hybrid_realizations, cutoff)
    block_estimates["profit_mean_sgsim"] = np.mean(profit_per_tonne(sgsim_realizations, cfg), axis=0)
    block_estimates["profit_mean_gcnn_rnn_sgsim"] = np.mean(profit_per_tonne(hybrid_realizations, cfg), axis=0)

    if "latent_truth" in blocks.columns:
        truth = blocks["latent_truth"].to_numpy(float)

        skill_sgsim = summarize_probabilistic_skill(sgsim_realizations, truth, cutoff)
        skill_hybrid = summarize_probabilistic_skill(hybrid_realizations, truth, cutoff)

        point_sgsim = {
            "rmse": float(np.sqrt(np.mean((block_estimates["sgsim_mean"].to_numpy(float) - truth) ** 2))),
            "mae": float(mean_absolute_error(truth, block_estimates["sgsim_mean"].to_numpy(float))),
            "r2": float(r2_score(truth, block_estimates["sgsim_mean"].to_numpy(float))),
        }
        point_hybrid = {
            "rmse": float(np.sqrt(np.mean((block_estimates["gcnn_rnn_sgsim_mean"].to_numpy(float) - truth) ** 2))),
            "mae": float(mean_absolute_error(truth, block_estimates["gcnn_rnn_sgsim_mean"].to_numpy(float))),
            "r2": float(r2_score(truth, block_estimates["gcnn_rnn_sgsim_mean"].to_numpy(float))),
        }

        truth_labels = (truth >= cutoff).astype(int)
        hybrid_probabilities = probability_of_ore(hybrid_realizations, cutoff)
        threshold = select_threshold_from_training(hybrid_probabilities, truth_labels)
        selected = (hybrid_probabilities >= threshold).astype(int)

        total_profit_hybrid = np.sum(profit_per_tonne(hybrid_realizations, cfg)[:, selected == 1], axis=1)
        total_profit_sgsim = np.sum(profit_per_tonne(sgsim_realizations, cfg)[:, selected == 1], axis=1)

        summary = pd.DataFrame([
            {
                "method": "SGSIM",
                **skill_sgsim,
                **point_sgsim,
                "expected_total_profit": float(np.mean(total_profit_sgsim)),
                "cvar_5": float(np.mean(total_profit_sgsim[total_profit_sgsim <= np.quantile(total_profit_sgsim, 0.05)])),
            },
            {
                "method": "GCNN-RNN-SGSIM",
                **skill_hybrid,
                **point_hybrid,
                "expected_total_profit": float(np.mean(total_profit_hybrid)),
                "cvar_5": float(np.mean(total_profit_hybrid[total_profit_hybrid <= np.quantile(total_profit_hybrid, 0.05)])),
            },
        ])
    else:
        truth_labels = (samples["grade"].to_numpy(float) >= cutoff).astype(int)
        train_probabilities = 1.0 / (1.0 + np.exp(-(drift_oof - np.median(drift_oof))))
        threshold = select_threshold_from_training(train_probabilities, truth_labels)

        calibrator = LogisticRegression(max_iter=2000)
        calibrator.fit(drift_oof.reshape(-1, 1), truth_labels)
        block_estimates["p_ore_gcnn_rnn_sgsim"] = calibrator.predict_proba(drift_block.reshape(-1, 1))[:, 1]

        summary = pd.DataFrame([
            {
                "method": "GCNN-RNN-SGSIM",
                "oof_rmse_log": float(np.sqrt(np.mean((drift_oof - samples["grade_log"].to_numpy(float)) ** 2))),
                "oof_mae_log": float(mean_absolute_error(samples["grade_log"].to_numpy(float), drift_oof)),
                "oof_r2_log": float(r2_score(samples["grade_log"].to_numpy(float), drift_oof)),
                "decision_threshold": float(threshold),
            }
        ])

    block_estimates.to_csv(Path(cfg.output_dir) / "block_estimates.csv", index=False)
    np.save(Path(cfg.output_dir) / "realizations.npy", hybrid_realizations)
    summary.to_csv(Path(cfg.output_dir) / "summary.csv", index=False)

    with open(Path(cfg.output_dir) / "config.json", "w", encoding="utf-8") as handle:
        json.dump(asdict(cfg), handle, indent=2)


def parse_args() -> Config:
    parser = argparse.ArgumentParser(description="Run GCNN-RNN-SGSIM.")
    parser.add_argument("--samples", default="", help="Path to the sample CSV file.")
    parser.add_argument("--blocks", default="", help="Path to the block CSV file.")
    parser.add_argument("--zones", default="", help="Path to the zone CSV file.")
    parser.add_argument("--output-dir", default="results", help="Directory for outputs.")
    parser.add_argument("--seed", type=int, default=7, help="Random seed.")
    parser.add_argument("--n-realizations", type=int, default=50, help="Number of conditional realizations.")
    args = parser.parse_args()

    cfg = Config()
    cfg.sample_path = args.samples
    cfg.block_path = args.blocks
    cfg.zone_path = args.zones
    cfg.output_dir = args.output_dir
    cfg.seed = args.seed
    cfg.n_realizations = args.n_realizations
    return cfg


if __name__ == "__main__":
